# Spindle Orientation and Positioning Simulation

---
## 0. User Parameters
**Edit this cell to control the demo.** All other cells can be run as-is.

In [ ]:
# ============================================================
# USER PARAMETERS — edit here
# ============================================================

# Simulation length. Shorter = faster but less convergence.
# Suggested: 30s for quick look, 300s for publication-quality
TOTAL_TIME = 60.0       # seconds
TIME_STEP  = 0.05       # seconds (do not change unless you know what you're doing)

# Number of astral MTs per pole (more = smoother but slower)
N_ASTRAL_MTS = 50

# Force parameters (pN)
PULL_FORCE = 5.0
PUSH_FORCE = 5.0

# Random seed for reproducibility
SEED = 42

# Save plots to disk? (True = slower, produces PDF frames)
SAVE_PLOTS = True

# Zebrafish: path to data files (required for zebrafish cell only)
# Download from: [repository link]
ZEBRAFISH_DATA_DIR     = "./data"
ZEBRAFISH_ENDO_INDEX   = 11       # which cell to simulate (1-indexed)

# ============================================================
print(f"Demo settings: {TOTAL_TIME}s simulation, {N_ASTRAL_MTS} MTs/pole")
print(f"Estimated runtime per cell type: ~{int(TOTAL_TIME/TIME_STEP * N_ASTRAL_MTS * 2 / 50000 * 10)}s")

---
## 1. Imports

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from numpy import linalg as LA

# Import simulation module (must be in same directory)
import importlib, spindle
importlib.reload(spindle)

from spindle import (
    FollicleEpithelialConfig, NeuroblastConfig,
    CElegansPNCConfig, CElegansSpindleConfig, ZebrafishEndoConfig,
    simulate
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print("Imports OK")

---
## 2. Follicular Epithelial Cell (*Drosophila*)

In *Drosophila* follicular epithelium, the spindle aligns planar (parallel to the epithelial sheet). This is driven by force generators concentrated at the **basal and lateral** cortex, with no apical FGs. The model reproduces this alignment from an arbitrary initial spindle angle using only the asymmetric FG distribution.

In [ ]:
config_follicle = FollicleEpithelialConfig(
    time_step          = TIME_STEP,
    total_time         = TOTAL_TIME,
    seed               = SEED,
    n_astral_mts       = N_ASTRAL_MTS,
    pull_force         = PULL_FORCE,
    push_force         = PUSH_FORCE,
    spindle_half_length= 0.4,
    spindle_width      = 0.32,
    spindle_angle_init = np.deg2rad(90),   # start vertical, should align to 0°
    fg_density         = 100,
    mt_mean_length     = 1.0,
    mt_length_stdev    = 1.0,
    save_plots         = SAVE_PLOTS,
    plot_interval      = 40,
)

results_follicle = simulate(config_follicle, run_id=2)
print(f"Final angle: {results_follicle['angles'][-1]:.1f}°")

---
## 3. Neuroblast (*Drosophila*)

*Drosophila* neuroblasts divide asymmetrically, with the spindle oriented along the apical-basal axis. This is driven by **apical** cortical FGs (recruiting LGN/Mud and dynein). The model captures apical-directed spindle alignment by concentrating FGs at the apical pole.

In [ ]:
config_neuroblast = NeuroblastConfig(
    time_step           = TIME_STEP,
    total_time          = TOTAL_TIME,
    seed                = SEED,
    n_astral_mts        = N_ASTRAL_MTS,
    pull_force          = PULL_FORCE,
    push_force          = PUSH_FORCE,
    spindle_half_length = 0.4,
    spindle_width       = 0.32,
    spindle_angle_init  = np.deg2rad(0),   # start horizontal, should align to 90°
    fg_density_basal    = 100,
    fg_density_apical   = 100,
    mt_mean_length      = 1.0,
    mt_length_stdev     = 1.0,
    save_plots          = SAVE_PLOTS,
    plot_interval       = 40,
)

results_neuroblast = simulate(config_neuroblast, run_id=1)
print(f"Final angle: {results_neuroblast['angles'][-1]:.1f}°")

---
## 4. *C. elegans* Embryo

The *C. elegans* one-cell embryo is a classic model for spindle positioning. The spindle must:
1. **Align** along the long (anterior-posterior) axis — reproduced in the PNC mode
2. **Displace posteriorly** — driven by asymmetric (posterior-enriched) FGs

The cell geometry is a superellipse (more rectangular than an ellipse), matching the actual embryo shape. MT dynamic instability parameters are tuned to published values.

In [ ]:
# --- PNC stage: spindle aligns along A-P axis ---
config_pnc = CElegansPNCConfig(
    time_step            = TIME_STEP,
    total_time           = TOTAL_TIME,
    seed                 = SEED,
    n_astral_mts         = N_ASTRAL_MTS,
    pull_force           = PULL_FORCE,
    push_force           = PUSH_FORCE,
    spindle_half_length  = 0.5,
    spindle_width        = 0.5,
    fg_density_anterior  = 40,
    fg_density_posterior = 60,
    mt_mean_length       = 9.0,
    mt_length_stdev      = 0.167,
    save_plots           = SAVE_PLOTS,
    plot_interval        = 40,
)

results_pnc = simulate(config_pnc, run_id=1)
print(f"PNC final angle: {results_pnc['angles'][-1]:.1f}°")

In [ ]:
# --- Spindle stage: posterior displacement ---
config_spindle = CElegansSpindleConfig(
    time_step            = TIME_STEP,
    total_time           = TOTAL_TIME,
    seed                 = SEED,
    n_astral_mts         = N_ASTRAL_MTS,
    pull_force           = PULL_FORCE,
    push_force           = PUSH_FORCE,
    spindle_half_length  = 0.9,
    spindle_width        = 0.5,
    fg_density_anterior  = 40,
    fg_density_posterior = 60,
    mt_mean_length       = 9.0,
    mt_length_stdev      = 0.167,
    save_plots           = SAVE_PLOTS,
    plot_interval        = 40,
)

results_spindle_ce = simulate(config_spindle, run_id=1)
print(f"Spindle stage final center displacement: {LA.norm(results_spindle_ce['centers'][-1]):.3f} model units")

---
## 5. Zebrafish Endothelial Cell

Unlike the idealized geometries above, zebrafish endothelial cells are imaged live and their actual contours are extracted from microscopy images frame-by-frame. The cell shape changes over time as the simulation runs.

### ⚠️ Data files required
This section requires the microscopy data files. Please download them from the repository and place them as follows:
```
./data/
    Movie_info.xlsx
    cells/
        cell_11/
            Mask_1.jpg, Mask_2.jpg, ...
    spindles/
        spindle_11/
            Spindle_1.jpg, ...
```
Then set `ZEBRAFISH_DATA_DIR` and `ZEBRAFISH_ENDO_INDEX` in the User Parameters cell above.

In [ ]:
data_dir = Path(ZEBRAFISH_DATA_DIR)
movie_info = data_dir / "Movie_info.xlsx"

if not movie_info.exists():
    print("⚠️  Data files not found. Skipping zebrafish simulation.")
    print(f"   Expected: {movie_info.resolve()}")
    print("   Please download the data files and update ZEBRAFISH_DATA_DIR above.")
    results_zebrafish = None
else:
    config_zebrafish = ZebrafishEndoConfig(
        time_step           = TIME_STEP,
        total_time          = TOTAL_TIME,
        seed                = SEED,
        n_astral_mts        = N_ASTRAL_MTS,
        pull_force          = PULL_FORCE,
        push_force          = PUSH_FORCE,
        fg_density          = 10,
        mt_mean_length      = 1.0,
        mt_length_stdev     = 1.0,
        endo_index          = ZEBRAFISH_ENDO_INDEX,
        fg_distribution     = "uniform",
        movie_info_file     = movie_info,
        cell_images_dir     = data_dir / "cells",
        spindle_images_dir  = data_dir / "spindles",
        save_plots          = SAVE_PLOTS,
        plot_interval       = 40,
    )

    results_zebrafish = simulate(config_zebrafish, run_id=1)
    print(f"Final angle: {results_zebrafish['angles'][-1]:.1f}°")